# 01 · Isotope dataset

Code for: El Farchouni, A., et al. *Groundwater Recharge Mechanisms in the Haouz-Mejjate Aquifer (Morocco): Insights from Environmental Tracers and Self-Organizing Maps.* Groundwater for Sustainable Development, 101706. https://doi.org/10.1016/j.gsd.2026.101706

**What this notebook does**
1. Loads the compiled dataset (540 samples) and keeps the metadata and the stable-isotope columns (δ¹⁸O, δ²H).
2. Keeps the **342 samples with both δ¹⁸O and δ²H** (plus three groundwater samples with a tritium analysis only, used in the chloride screening of the EMMA notebook).
3. Harmonizes the water-type labels and computes the deuterium excess, d-excess = δ²H − 8 δ¹⁸O (Dansgaard, 1964).
4. Exports the isotope dataset used by `02_isotopes.ipynb` (Figure 10) and by the EMMA notebooks.

| | |
|---|---|
| **Input** | `data/All_data1.xlsx` |
| **Output** | `outputs/intermediate/data_EMMA1.xlsx` (345 samples) |
| **Next notebook** | `02_isotopes.ipynb` |

*Note: the tritium (³H) column is not part of the shared data file (available on request), so the tritium cross-check of the article (Figure SM13) is not reproduced in this repository.*

### <font color='orange'> **1) Packages, settings and paths** </font>

In [1]:
# ── Core ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# ── Display settings ─────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.width', 120)

# ── Plot defaults ────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'figure.figsize': (10, 6),
    'savefig.bbox': 'tight',
    'savefig.transparent': False,
})

# ── Paths (built from the repository root, so they work on any computer) ──
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data").is_dir() and (p / "notebooks").is_dir())
DATA_FILE    = ROOT / "data" / "All_data1.xlsx"
DIR_INTERMED = ROOT / "outputs" / "intermediate"
DIR_TABLES   = ROOT / "outputs" / "tables"
DIR_FIGURES  = ROOT / "outputs" / "figures"
for d in (DIR_INTERMED, DIR_TABLES, DIR_FIGURES):
    d.mkdir(parents=True, exist_ok=True)
print(f"Data file : {DATA_FILE.relative_to(ROOT)}  (exists: {DATA_FILE.is_file()})")

C:\Users\Admin\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Data file : data\All_data1.xlsx  (exists: True)


### <font color='orange'> **2) Load the data and select the isotope columns** </font>

In [2]:
# ── Load the compiled dataset ────────────────────────────────────────
df_raw = pd.read_excel(DATA_FILE, sheet_name="Sheet1")

print(f"Dataset shape: {df_raw.shape[0]} samples × {df_raw.shape[1]} columns\n")

# ── Column overview ──────────────────────────────────────────────────
print(f"{'#':>3}  {'Column':<25s} {'Dtype':<15s} {'Non-null':>8s}  {'Missing':>7s}")
print("─" * 65)
for i, col in enumerate(df_raw.columns):
    nn = df_raw[col].notna().sum()
    na = df_raw[col].isna().sum()
    print(f"{i:>3}  {col:<25s} {str(df_raw[col].dtype):<15s} {nn:>8d}  {na:>7d}")

Dataset shape: 540 samples × 29 columns

  #  Column                    Dtype           Non-null  Missing
─────────────────────────────────────────────────────────────────
  0  Reference                 object               540        0
  1  SampleID                  object               540        0
  2  X                         float64              454       86
  3  Y                         float64              454       86
  4  Z                         float64              308      232
  5  depth to water (m)        float64              266      274
  6  Total Depth(m)            object                65      475
  7  Sampling date             object               540        0
  8  Sample                    object                74      466
  9  Type                      object               540        0
 10  Ca (mg/L)                 float64              410      130
 11  Mg (mg/L)                 float64              407      133
 12  Na (mg/L)                 float64          

In [3]:
# ── Select the columns needed for the isotope and EMMA notebooks ─────
KEEP_COLS = [
    # Metadata
    "Reference", "SampleID", "X", "Y", "Z",
    "Sampling date", "Type",
    # Isotope tracers
    "δ18O (‰ VSMOW)", "δ2H (‰ VSMOW)",
]

df = df_raw[KEEP_COLS].copy()

print(f"New dataframe: {df.shape[0]} samples × {df.shape[1]} columns\n")
print(f"{'Column':<25s} {'Non-null':>8s}  {'Missing':>7s}")
print("─" * 45)
for col in df.columns:
    nn = df[col].notna().sum()
    na = df[col].isna().sum()
    print(f"{col:<25s} {nn:>8d}  {na:>7d}")

df.head()

New dataframe: 540 samples × 9 columns

Column                    Non-null  Missing
─────────────────────────────────────────────
Reference                      540        0
SampleID                       540        0
X                              454       86
Y                              454       86
Z                              308      232
Sampling date                  540        0
Type                           540        0
δ18O (‰ VSMOW)                 342      198
δ2H (‰ VSMOW)                  342      198


,Reference,SampleID,X,Y,Z,Sampling date,Type,δ18O (‰ VSMOW),δ2H (‰ VSMOW)
0,"(EL FARCHOUNI et al., 2025)",S1,264704.0,468828.0,2809.70,2025-01-10 00:00:00,Snow,-9.91,-63.4
1,"(EL FARCHOUNI et al., 2025)",S2,264894.0,469092.0,2728.25,2025-01-10 00:00:00,Snow,-11.44,-72.8
2,"(EL FARCHOUNI et al., 2025)",S3,264933.0,469332.0,2703.40,2025-01-10 00:00:00,Snow,-10.13,-63.0
3,"(EL FARCHOUNI et al., 2025)",S4,266042.0,468235.0,2718.90,2025-01-10 00:00:00,Snow,-10.56,-64.9
4,"(EL FARCHOUNI et al., 2025)",S5,266236.0,469982.0,2646.00,2025-01-10 00:00:00,Lac,-7.84,-51.5


#### **2.1 Keep the samples with isotope data**

> The 342 samples with δ¹⁸O and δ²H, plus three groundwater samples that have a tritium analysis only (S406, S419, S437; Boukhari et al., 2015). The tritium values are not in the shared file, but these samples are kept so that the chloride-screening set of the EMMA notebook is the same as in the article (214 samples with Na⁺ and Cl⁻).

In [4]:
# ── Keep samples with stable isotopes, plus the tritium-only samples ──
isotope_cols = ["δ18O (‰ VSMOW)", "δ2H (‰ VSMOW)"]
TRITIUM_ONLY = ["S406", "S419", "S437"]   # groundwater samples with a 3H analysis only (Boukhari et al., 2015)

before = len(df)
keep = df[isotope_cols].notna().all(axis=1) | df["SampleID"].isin(TRITIUM_ONLY)
df = df[keep].reset_index(drop=True)
after = len(df)

print(f"Before: {before} samples")
print(f"Removed: {before - after} samples without isotope data")
print(f"After:  {after} samples kept ({df[isotope_cols].notna().all(axis=1).sum()} with δ18O and δ2H, {len(TRITIUM_ONLY)} with tritium only)")

Before: 540 samples
Removed: 195 samples without isotope data
After:  345 samples kept (342 with δ18O and δ2H, 3 with tritium only)


#### **2.2 Water-type labels**

In [5]:
# ── Rename water types ────────────────────────────────────────────────
df["Type"] = df["Type"].replace({
    "Tributary": "Stream",
    "Zaraba":    "Canal (Zaraba)",
})

print("Water types after renaming:")
print(df["Type"].value_counts().to_string())

Water types after renaming:
Type
Groundwater       161
Spring             69
Stream             41
Snow               37
Rain               21
Lac                11
Dam                 2
Irrigation          2
Canal (Zaraba)      1


#### **2.3 Deuterium excess**

d-excess = δ²H − 8 × δ¹⁸O (Dansgaard, 1964).

In [6]:
# ── Compute d-excess ─────────────────────────────────────────────────
df["d-excess (‰)"] = df["δ2H (‰ VSMOW)"] - 8 * df["δ18O (‰ VSMOW)"]

nn = df["d-excess (‰)"].notna().sum()
print(f"d-excess computed for {nn} / {len(df)} samples")
print(f"  (NaN where δ18O or δ2H is missing)\n")

print(df[["SampleID", "δ18O (‰ VSMOW)", "δ2H (‰ VSMOW)", "d-excess (‰)"]].head(10))

d-excess computed for 342 / 345 samples
  (NaN where δ18O or δ2H is missing)

  SampleID  δ18O (‰ VSMOW)  δ2H (‰ VSMOW)  d-excess (‰)
0       S1           -9.91          -63.4         15.88
1       S2          -11.44          -72.8         18.72
2       S3          -10.13          -63.0         18.04
3       S4          -10.56          -64.9         19.58
4       S5           -7.84          -51.5         11.22
5       S6           -7.94          -51.2         12.32
6       S7           -6.81          -42.1         12.38
7       S8           -7.96          -51.0         12.68
8       S9           -7.75          -49.6         12.40
9      S10           -8.08          -51.8         12.84


### <font color='orange'> **3) Export** </font>

In [7]:
# ── Export to Excel ───────────────────────────────────────────────────
output_file = DIR_INTERMED / "data_EMMA1.xlsx"
df.to_excel(output_file, index=False)

print(f"✓ Exported: {output_file.relative_to(ROOT)}")
print(f"  Shape: {df.shape[0]} samples × {df.shape[1]} columns")

# ── Check against the article ────────────────────────────────────────
assert len(df) == 345 and df[isotope_cols].notna().all(axis=1).sum() == 342
print("\n✓ Matches the article: 342 samples with δ18O and δ2H (+ 3 tritium-only samples for the chloride screening).")

✓ Exported: outputs\intermediate\data_EMMA1.xlsx
  Shape: 345 samples × 10 columns

✓ Matches the article: 342 samples with δ18O and δ2H (+ 3 tritium-only samples for the chloride screening).


### <font color='orange'> **4) Reference lines used in the article** </font>

| Line | Equation | Scale | Source |
|------|----------|-------|--------|
| **GMWL** | δ²H = 8 δ¹⁸O + 10 | global | Craig (1961) |
| **LMWL** (Morocco) | δ²H = 7.73 δ¹⁸O + 9.18 | national | Ait Brahim et al. (2016) |
| **LEL** | fitted in `02_isotopes.ipynb` on the evaporation-affected samples | this study | |
| **LAL** | δ¹⁸O = −0.0027 × Z (gradient −0.27 ‰ per 100 m, Bouchaou et al., 1995) | Ourika catchment | Rhoujjati et al. (2021) |
| regional gradient | −0.18 ‰ per 100 m, used for the uncertainty of the recharge altitude | Morocco | Ait Brahim et al. (2016) |

---
**Output:** `outputs/intermediate/data_EMMA1.xlsx` (345 samples: 342 with δ¹⁸O and δ²H, 3 with tritium only)

**Next:** `02_isotopes.ipynb` (Figure 10)